# Load System Data and Compute Average Execution Accuracy (EX) per System

In [16]:
# Read the System data
df = pd.read_csv('/data/evaluation/experiments/systems_evel_on_bird/original_systems_data_with_metrics.csv')

# check null values in all columns
null_counts = df.isnull().sum()
print(null_counts)

# check "error" and "timout" values in

system                             0
db_name                            0
question_id                        0
question                           0
gold_sql                           0
predicted_sql                      0
EXECUTION_ACCURACY                 0
EXACT_COLUMN_AND_EXACT_CELL        0
SEMANTIC_COLUMN_AND_EXACT_CELL     0
UNIFIED_COLUMN_AND_SEMANTIC_ROW    3
dtype: int64


# System average EX

In [11]:
import pandas as pd
import json

# Read the System data
df = pd.read_csv('/data/evaluation/experiments/systems_evel_on_bird/original_systems_data_with_metrics.csv')

def extract_ex(val):
    try:
        if isinstance(val, str):
            val = json.loads(val)
        return val.get('EX', 0) if isinstance(val, dict) else 0
    except Exception:
        return 0

df['EX'] = df['EXECUTION_ACCURACY'].apply(extract_ex)
avg_ex_per_system = df.groupby('system')['EX'].mean().reset_index()
print(avg_ex_per_system)

    system        EX
0    chess  0.650587
1  din-sql  0.589309
2  mac-sql  0.559035


# Filter Rows with EX == 1 and Technique Failures (EXP or EXR != 1)

In [14]:
# filters rows where the EX value in EXECUTION_ACCURACY is 1, and then further filters those rows to keep only ones where at least one technique column has EXP or EXR not equal to 1.

import pandas as pd
import json

# Read the CSV
df = pd.read_csv('/data/evaluation/experiments/systems_evel_on_bird/original_systems_data_with_metrics.csv')

# Parse the JSON columns
technique_cols = [
    'EXACT_COLUMN_AND_EXACT_CELL',
    'SEMANTIC_COLUMN_AND_EXACT_CELL',
    'UNIFIED_COLUMN_AND_SEMANTIC_ROW'
]
for col in ['EXECUTION_ACCURACY'] + technique_cols:
    df[col] = df[col].apply(json.loads)

# Filter EX == 1
filtered = df[df['EXECUTION_ACCURACY'].apply(lambda x: x.get('EX', 0) == 1)]

# Check if any EXP or EXR != 1 in any technique column
def exp_exr_not_1(row):
    for col in technique_cols:
        exp = row[col].get('EXP')
        exr = row[col].get('EXR')
        if exp != 1 or exr != 1:
            return True
    return False

result = filtered[filtered.apply(exp_exr_not_1, axis=1)]

print(len(result))

result.to_csv('/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/evaluation/experiments/systems_evel_on_bird/filtered_ex_1_with_technique_failures.csv', index=False)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

# Find Intersection of Failed Questions (EX == 0) Across All Systems

show case sample in paper for "why exact column and exact cell is good" chess california_schools 87

In [3]:
# ilters rows where the execution accuracy (EX) is 0 (indicating failure), finds the set of question_ids that failed in all systems (intersection)

import pandas as pd
import json

# Read the CSV
df = pd.read_csv('/data/evaluation/experiments/systems_evel_on_bird/systems_data_with_metrics.csv')

# Parse EXECUTION_ACCURACY column
df['EXECUTION_ACCURACY'] = df['EXECUTION_ACCURACY'].apply(json.loads)

# Filter rows where EX == 0
failures = df[df['EXECUTION_ACCURACY'].apply(lambda x: x.get('EX', 0) == 0)]

# Find intersection of question_ids across all systems
systems = failures['system'].unique()
question_ids_per_system = [set(failures[failures['system'] == sys]['question_id']) for sys in systems]
common_question_ids = set.intersection(*question_ids_per_system)

# Filter failures to keep only rows with common question_ids
intersection_failures = failures[failures['question_id'].isin(common_question_ids)]

print(len(intersection_failures))
intersection_failures.to_csv('/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/evaluation/experiments/systems_evel_on_bird/intersection_failures.csv', index=False)

6


In [7]:
import pandas as pd

# Read the CSV file
df = pd.read_csv(
    '/data/evaluation/experiments/systems_evel_on_bird/systems_data_with_metrics1.csv')

# Remove newlines from predicted_query column and replace with space
df['predicted_sql'] = df['predicted_sql'].str.replace('\n', ' ')

# Save the modified dataframe back to CSV
df.to_csv(
    '/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/evaluation/experiments/systems_evel_on_bird/systems_data_with_ex.csv',
    index=False)


# Update EXP and EXR F1 to 1 if EX == 1

In [21]:
import pandas as pd
import json

# Read the CSV file
df = pd.read_csv('/data/evaluation/experiments/systems_evel_on_bird/original_systems_data_with_metrics.csv')

technique_cols = ['EXACT_COLUMN_AND_EXACT_CELL', 'SEMANTIC_COLUMN_AND_EXACT_CELL', 'UNIFIED_COLUMN_AND_SEMANTIC_ROW']

def safe_json_loads(val):
    if pd.isna(val) or not isinstance(val, str):
        return None
    try:
        return json.loads(val)
    except (json.JSONDecodeError, TypeError):
        return None

def update_technique_metrics(row):
    exec_acc = safe_json_loads(row['EXECUTION_ACCURACY'])
    if exec_acc is None:
        return row
    if exec_acc.get('EX') == 1:
        for col in technique_cols:
            technique_data = safe_json_loads(row[col])
            if technique_data is None:
                continue
            technique_data['EX'] = 1
            technique_data['EXP'] = 1.0
            technique_data['EXR'] = 1.0
            technique_data['F1'] = 1.0
            technique_data['latency'] = exec_acc.get("latency")
            row[col] = json.dumps(technique_data)
    return row

df = df.apply(update_technique_metrics, axis=1)
df.to_csv('/Users/mhmalekpour/PycharmProjects/text-to-sql-coverage/data/evaluation/experiments/systems_evel_on_bird/systems_data_with_metrics_updated.csv', index=False)